In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm   # optional - pip install tqdm if missing

# ────────────────────────────────────────────────
# CONFIG
# ────────────────────────────────────────────────
FILE_PATH = "Copy of Master Data_290102026 2 - Copy.xlsx"
SHEET_NAME = "Master Data "

MAX_HOURS_PER_DAY = 22.0
CHANGEOVER_HOURS = 40 / 60.0
TARGET_COVERAGE_DAYS = 3.0
MAX_PARTS_PER_MACHINE = 3

RUNNER_THRESHOLD = 2000
REPEATER_THRESHOLD = 200

ALLOWED_MACHINES = [
    "MP-01", "MP-04", "MP-05", "MP-08", "MP-10",
    "MP-11", "MP-17", "TOYO-IST", "TOYO1ST", "TOYO IST"
]

# ────────────────────────────────────────────────
# MACHINE NAME CLEANING
# ────────────────────────────────────────────────
def clean_machine(raw):
    if pd.isna(raw) or not str(raw).strip():
        return None
    s = str(raw).strip().upper()
    s = s.replace(".", "-").replace("M.P-", "MP-").replace("MP.", "MP-")
    s = s.replace("TOYOI ST", "TOYO-IST").replace("TOYO IST", "TOYO-IST").replace("TOYO1ST", "TOYO-IST")
    s = s.replace(" ", "-")
    return s

def parse_machines(cell):
    if pd.isna(cell):
        return []
    parts = str(cell).split(",")
    cleaned = [clean_machine(x) for x in parts if clean_machine(x)]
    return list(set(cleaned))

# ────────────────────────────────────────────────
# 1. READ & AGGREGATE (this is the heavy part — but only once)
# ────────────────────────────────────────────────
print("Reading large Excel file... (may take 20–90 seconds)")
master = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

print("Cleaning numeric columns...")
for col in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time"]:
    master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0)

print("Aggregating by Child Part... (grouping ~118k rows)")
agg_list = []
for child, g in master.groupby("Child Part"):
    daily_d = (g["Daily Plan"] * g["Sub Count"]).sum()
    if daily_d <= 0:
        continue  # skip useless parts early

    agg_list.append({
        "Child Part": child,
        "Daily_Demand": daily_d,
        "Min_Qty": g["Minimum Quantity"].iloc[0],
        "Inventory": g["Inventory_25"].iloc[0],
        "Cycle_Time_sec": g["Cycle Time"].iloc[0],
        "Vertical_Machines_raw": ",".join(g["Vertical Machines"].dropna().unique().astype(str))
    })

df = pd.DataFrame(agg_list)
df["Net_Required"] = df["Daily_Demand"] + df["Min_Qty"] - df["Inventory"]
df["Net_Required"] = df["Net_Required"].clip(lower=0)

print(f"Aggregation complete → {len(df):,} unique Child Parts")

# Classify
df["Category"] = "Stranger"
df.loc[df["Daily_Demand"] >= RUNNER_THRESHOLD, "Category"] = "Runner"
df.loc[(df["Daily_Demand"] >= REPEATER_THRESHOLD) & (df["Daily_Demand"] < RUNNER_THRESHOLD), "Category"] = "Repeater"

print("\nClassification:")
print(df["Category"].value_counts())

# Only schedule non-runners with demand
to_schedule = df[(df["Category"].isin(["Repeater", "Stranger"])) & (df["Daily_Demand"] > 0)].copy()
to_schedule["Eligible_Machines"] = to_schedule["Vertical_Machines_raw"].apply(parse_machines)

print(f"\nParts to schedule: {len(to_schedule):,}")
print("Sample eligible machines:")
print(to_schedule[["Child Part", "Daily_Demand", "Eligible_Machines"]].head(10))

# ────────────────────────────────────────────────
# SCHEDULER (now only on ~thousands of rows)
# ────────────────────────────────────────────────
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_parts = {m: [] for m in ALLOWED_MACHINES}
schedule = []

print("\nStarting Phase 1 (daily demand) – may take a few seconds to minutes...")
for _, part in tqdm(to_schedule.iterrows(), total=len(to_schedule), desc="Phase 1"):
    if part["Daily_Demand"] <= 0:
        continue

    hrs_per_pc = part["Cycle_Time_sec"] / 3600.0
    if hrs_per_pc <= 0:
        continue

    qty_left = part["Daily_Demand"]
    eligible = [m for m in part["Eligible_Machines"] if m in ALLOWED_MACHINES]
    if not eligible:
        continue

    for m in sorted(eligible, key=machine_load.get):
        free = MAX_HOURS_PER_DAY - machine_load[m]
        if free <= CHANGEOVER_HOURS:
            continue

        setup_h = CHANGEOVER_HOURS  # conservative (no color info)
        free_after = free - setup_h
        if free_after <= 0:
            continue

        max_qty = free_after / hrs_per_pc
        assign_qty = min(qty_left, max_qty)
        if assign_qty < 5:
            continue

        assign_h = assign_qty * hrs_per_pc
        machine_load[m] += assign_h + setup_h
        machine_parts[m].append({"Part": part["Child Part"], "Qty": assign_qty, "Type": "Daily"})

        qty_left -= assign_qty
        if qty_left <= 0:
            break

# Phase 2 can be added later once Phase 1 works
print("\nPhase 1 finished. Current load:")
for m, h in sorted(machine_load.items(), key=lambda x: x[1], reverse=True):
    if h > 0:
        print(f"{m:8} → {h:5.1f} h")